# Figure 3: Temperature

Standalone paired-array workflow. Plotting is kept in the final cell and follows public `Main_figure.ipynb` cells 20 and 25. The default uses the frozen numeric reference; set `INPUT_MODE = "raw"` to load TIFFs and compute.


In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "config" / "analysis.json").exists():
    if REPO_ROOT.parent == REPO_ROOT:
        raise FileNotFoundError("Could not locate the analysis repository root")
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / "src"))

from elephant_analysis.runtime import configure_runtime
configure_runtime()

import numpy as np
import matplotlib.pyplot as plt
from elephant_analysis.data_inputs import load_inputs
from elephant_analysis.artifacts import save_arrays, load_arrays
from elephant_analysis.array_utils import nanravel
from elephant_analysis.paired_sites import compute_paired

DATA_ROOT = REPO_ROOT / "data"
RESULTS_ROOT = REPO_ROOT / "results"
FIGURES_ROOT = REPO_ROOT / "figures"
FIGURES_ROOT.mkdir(parents=True, exist_ok=True)
INPUT_MODE = "reference"
RUN_ANALYSIS = True


In [ ]:
cache_path = RESULTS_ROOT / "paired" / "paired_results.npz"
if INPUT_MODE not in ("reference", "raw"):
    raise ValueError("INPUT_MODE must be reference or raw")
if RUN_ANALYSIS:
    loaded = load_inputs("paired", DATA_ROOT, mode=INPUT_MODE)
    paired_arrays = compute_paired(loaded) if INPUT_MODE == "raw" else loaded
    save_arrays(cache_path, **paired_arrays)
    paired_arrays = load_arrays(cache_path)
# Cached mode loads the figure-specific result in the next calculation cell.


## Paired inputs


In [ ]:
from elephant_analysis.figure_statistics import prepare_temperature_plot_data
plot_path = RESULTS_ROOT / "fig3/temperature_plot_data.npz"
if RUN_ANALYSIS:
    save_arrays(plot_path, **prepare_temperature_plot_data(paired_arrays))
plot_data = load_arrays(plot_path)
df_delta = [plot_data["lst_1516"], plot_data["lst_1214"]]
del_tree = [plot_data["th"], plot_data["tc"]]
del_lst = plot_data["lst"]


## Figure


In [ ]:
# Original plotting cells copied from Main_figure.ipynb.
# Keep source sizes, font settings, labels, colours, positions and line widths.
EXPORT_PDF = True
EXPORT_DPI = 600
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import kstest, gaussian_kde
from elephant_analysis import array_utils as gc
mpl.rcdefaults()
mpl.rcParams["pdf.fonttype"] = 42

def export_original(figure, stem, dpi=EXPORT_DPI):
    # Match the source savefig contract, including tight bounding boxes.
    FIGURES_ROOT.mkdir(parents=True, exist_ok=True)
    figure.savefig(FIGURES_ROOT / (stem + '.png'), dpi=dpi, bbox_inches='tight')
    if EXPORT_PDF:
        figure.savefig(FIGURES_ROOT / (stem + '.pdf'), bbox_inches='tight')

# BEGIN ORIGINAL CELL 20

from elephant_analysis import array_utils as gc
from scipy.stats import kstest
from scipy.stats import gaussian_kde
import matplotlib.colors as mcolors
import matplotlib
import matplotlib.pyplot as plt
%matplotlib inline
%config InlineBackend.figure_format = 'retina'
fig = plt.figure(figsize=(12,4))
plt.subplots_adjust(wspace=0.25,hspace=0.02)

font = {'family': 'sans-serif',
        'sans-serif': 'Arial',
        'weight': 'normal',
        'size': 14}
plt.rc('font', **font)

cl1= np.array([(211,145,161)])
cl2= np.array([(231,195,204)])
cl=[cl1/255,cl1/255]

label1=['Drought area']
label2=['Drought area']
label=[label1,label2]

for i in range(2):
    ax = fig.add_subplot(1,2,i+1)
    x=np.linspace(-2,2,1000)
    mean=np.nanmean(df_delta[i])
    delta_rav=gc.nanravel(df_delta[i])
    kenal=gaussian_kde(delta_rav)
    z=kenal.evaluate(x)
    z_mean=kenal.evaluate(mean)
    ax.plot(x,z,lw=3,color=cl[0],label=label[i])
    ax.fill_between(x,0,z,facecolor=cl[i],alpha=0.2)
    ax.vlines(mean,0,z_mean,lw=2,ls='--',color=cl[0])
    ax.vlines(0,0,0.8,lw=2,ls='--',color='black')
    ax.spines['right'].set_visible(False)
    ax.spines['top'].set_visible(False)
    p=kstest(gc.nanravel(df_delta[i]), 'norm')[1]
    print(mean,p,len(delta_rav))
    if p<0.001:

        ax.text(0.75,0.89, 'p<0.001', fontsize=14,transform = ax.transAxes,color=cl[0])
    elif p<0.01:
        ax.text(0.75,0.89, 'p<0.01', fontsize=14,transform = ax.transAxes,color=cl[0])
    elif p<0.05:
        ax.text(0.75,0.89, 'p<0.05', fontsize=14,transform = ax.transAxes,color=cl[0])
    else:
        ax.text(0.75,0.89, 'p={:.2f}'.format(p), fontsize=14,transform = ax.transAxes,color=cl[0]) 

    if i ==0:
        ax.set_xlabel('$\Delta$LST during 2015-2016 (°C)',labelpad=10)
    else:
        ax.set_xlabel('$\Delta$LST during 2012-2014 (°C)',labelpad=10)

    ax=plt.gca()
    ax.yaxis.set_ticks_position('left')
    ax.spines['left'].set_position(('data',-1.6))
    ax.xaxis.set_ticks_position('bottom')
    ax.spines['bottom'].set_position(('data',-0.08))
    ax.set_ylim(0,0.6)
    ax.set_xlim(-1.5,1.5)
    ax.set_xticks(np.arange(-1.5,1.501,0.5))
    ax.set_yticks(np.arange(0,0.601,0.2))
    ax.set_ylabel('Probability density',labelpad=10)

    ax.tick_params(labelsize=14)
    ax.xaxis.label.set_size(18)
    ax.yaxis.label.set_size(18)
    ax.xaxis.set_tick_params(width=2)
    ax.yaxis.set_tick_params(width=2)
    ax.spines['left'].set_linewidth(2)
    ax.spines['bottom'].set_linewidth(2)

    pos = ax.get_position()

    ax1 = fig.add_axes([pos.x0, pos.y0 - 0.11, pos.width, 0.12],sharex=ax)
    ax1.axis('off')
    ax1.spines['top'].set_visible(False)
    ax1.spines['right'].set_visible(False)
    ax1.spines['left'].set_visible(False)
    ax1.tick_params(bottom=False,top=False, left=False, right=False)

    bplot=ax1.boxplot(gc.nanravel(df_delta[i]),
                      vert=False,
                      whis=(10,90),                
                      widths=0.3,
                      patch_artist=True,
                      showmeans=False,
                      meanprops = {'marker':'o','markerfacecolor':cl[i],"markeredgecolor":cl[i],"markersize":15,"alpha":0},
                      medianprops={'linewidth':'0.5',"color":cl[i],"alpha":0},
                      boxprops={"facecolor": 'none', "edgecolor": cl[i],"linewidth":2,"alpha":1},
                      capprops=None,
                      whiskerprops={'linewidth':'2','color':cl[i]},
                      showcaps=False,
                      showfliers = False,
                      flierprops = {'marker':'+','markersize':'3','markeredgecolor':cl[i],'color':cl[i]})
# END ORIGINAL CELL 20
export_original(fig, 'Fig3_temperature_distributions')
plt.show()

# BEGIN ORIGINAL CELL 25
import scipy
import statsmodels.api as sm
from scipy.stats import gaussian_kde
from sklearn.metrics import r2_score
from matplotlib.gridspec import GridSpec

%matplotlib inline
%config InlineBackend.figure_format = 'retina'
fig = plt.figure(figsize=(12,4))
fig.subplots_adjust(wspace=0.3,hspace=0, left=None, bottom=None, right=None, top=None)
font = {'family': 'sans-serif',
        'sans-serif': 'Arial',
        'weight': 'normal',
        'size': 14}
plt.rc('font', **font)

cl=['red','blue']
mark=['*','.']
x_s1=np.arange(-10,20)
x_s2=np.arange(-50,75)
x_s=[x_s1,x_s2]

cl1 = np.array([(82,160,134),(244,132,102)])
cl=cl1/255

for i in range(2):
    ax=fig.add_subplot(1,2,i+1)
    xy=np.vstack([del_tree[i],del_lst])
    z=gaussian_kde(xy)(xy)
    idx = z.argsort()
    x, y, z = np.array(del_tree[i])[idx], np.array(del_lst)[idx], z[idx]
    # Retain the manuscript/source rendering for visual consistency.
    im=ax.scatter(del_tree[i],del_lst,c=z,alpha=0.5,s=30,cmap='Greys')

    prefix = ('th', 'tc')[i]
    slope = float(plot_data[prefix + '_ols_slope'])
    ci_slope = plot_data[prefix + '_ols_slope_ci']
    xg = plot_data[prefix + '_fit_x']
    yhat = plot_data[prefix + '_fit_y']
    ci = plot_data[prefix + '_fit_ci']
    ax.plot(xg, yhat, color='red', lw=1.5, zorder=3, label='OLS fit')
    ax.fill_between(xg, ci[:,0], ci[:,1], color='deepskyblue', alpha=0.6, linewidth=1, zorder=2, label='95% CI')

    cor = np.corrcoef(del_tree[i], del_lst)[0,1]
    model_p = float(plot_data[prefix + '_ols_p'])
    p_label = ('p<0.001' if model_p < 0.001 else 'p<0.01' if model_p < 0.01
               else 'p<0.05' if model_p < 0.05 else 'p={:.2f}'.format(model_p))
    ax.text(0.75, 0.8, 'R\u00b2 = {:.2f}\n{}'.format(cor*cor, p_label),transform=ax.transAxes, color='red')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.set_ylim(-9,6.01)
    ax.set_yticks(np.arange(-9,6.01,3))
    if i ==0:
        ax.set_xlim(-10,20.01)
        ax.set_xticks(np.arange(-10,20.01,10))

        ax.set_xlabel('$Δ$Tree height (m)',size=18,labelpad=15)
        ax.set_ylabel('$Δ$LST (°C)',size=18,labelpad=5)
        ax=plt.gca()
        ax.yaxis.set_ticks_position('left')
        ax.spines['left'].set_position(('data',-11))
        ax.spines['left'].set_linewidth(2);

    if i==1:
        ax.set_xlim(-50,75.01)
        ax.set_xticks(np.arange(-50,75.01,25))

        ax.set_xlabel('$Δ$Tree cover (%)',size=18,labelpad=15)
        ax.set_ylabel('$Δ$LST (°C)',size=18,labelpad=5)
        ax=plt.gca()
        ax.yaxis.set_ticks_position('left')
        ax.spines['left'].set_position(('data',-54))
        ax.spines['left'].set_linewidth(2);
        
        position=fig.add_axes([0.35,-0.15,0.3,0.05])
        cbar = fig.colorbar(im,orientation='horizontal',cax=position)
        cbar.set_ticks(np.arange(0,0.031,0.01))
        cbar.ax.tick_params(labelsize=14)
        cbar.set_label(label='Data density',size=18)
        
    ax.xaxis.set_tick_params(width=2,labelsize=14,pad=5)
    ax.yaxis.set_tick_params(width=2,labelsize=14,pad=5)
    ax.spines['bottom'].set_linewidth(2);
# END ORIGINAL CELL 25
export_original(fig, 'Fig3_temperature_relationships')
plt.show()
